# Step 13: LSTM
An LSTM with 128 units reads the 25 frames of each second in order; its final memory goes to a small dense head (development stage, random split).
*Same results as `src/13_lstm.py`.*

**In short** (what this notebook shows)

- **What:** An LSTM on the 25 frames of each second.
- **Needs the NoXi data?** Yes. Random split (development step).
- **Result / use:** 0.857. Shows recurrent layers and gates.

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: **24 conversations** (English 026, 027, 028, 029, 030, 058, 062, 063; French 001, 007, 009, 011, 013, 014, 020, 021; German 068, 070, 072, 073, 074, 075, 076, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

**Imports.**

In [3]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
keras.utils.set_random_seed(42)

In [4]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Data:** 25 frames x 802 features per second, random split 60 / 20 / 20, every frame scaled (scaler fitted on training frames only).

In [5]:
y = pd.read_parquet("data/features/noxi_engagement.parquet", columns=["engagement"])["engagement"].to_numpy("float32")
X = np.load("data/features/noxi_sequences.npy", mmap_mode="r")   # read from disk piece by piece
rest, test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)
train, val = train_test_split(rest, test_size=0.25, random_state=42)
scaler = StandardScaler().fit(X[train].reshape(-1, 802))


def prepare(rows):
    return scaler.transform(X[rows].reshape(-1, 802)).reshape(len(rows), 25, 802).astype("float32")


X_train, X_val, X_test = prepare(train), prepare(val), prepare(test)
y_train, y_val, y_test = y[train], y[val], y[test]
del X

**The LSTM:** frame 1 -> frame 25 in order, 128 memory units, then dropout and a dense head.

In [6]:
model = keras.Sequential([
    layers.Input(shape=(25, 802)),
    layers.LSTM(128),
    layers.Dropout(0.1),
    layers.Dense(64, activation="relu"),
    layers.Dense(1),
], name="LSTM")
model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
model.summary()

Model: "LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 128)                 │         476,672 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 484,993 (1.85 MB)

 Trainable params: 484,993 (1.85 MB)

 Non-trainable params: 0 (0.00 B)

**Train 20 epochs** in 4 stages of 5, printing the validation CCC.

In [7]:
for stage in range(4):
    model.fit(X_train, y_train, epochs=5, batch_size=128, verbose=0)
    p = model.predict(X_val, verbose=0).flatten()
    print(f"  epoch {(stage + 1) * 5:3d}  val CCC = {ccc(y_val, p):.3f}")

  epoch   5  val CCC = 0.827


  epoch  10  val CCC = 0.843


  epoch  15  val CCC = 0.850


  epoch  20  val CCC = 0.852


**Results** next to the saved result (`reports/lstm_results.csv`).

In [8]:
val_ccc = ccc(y_val, model.predict(X_val, verbose=0).flatten())
pred = model.predict(X_test, verbose=0).flatten()
print(f"parameters {model.count_params():,} | validation CCC {val_ccc:.3f} | TEST CCC {ccc(y_test, pred):.3f} "
      f"| Pearson {np.corrcoef(y_test, pred)[0, 1]:.3f} | MAE {mean_absolute_error(y_test, pred):.3f}")
pd.DataFrame([dict(model="LSTM", params=model.count_params(), val_ccc=val_ccc, test_ccc=ccc(y_test, pred),
                   pearson=np.corrcoef(y_test, pred)[0, 1], mae=mean_absolute_error(y_test, pred))]).to_csv("reports/lstm_results.csv", index=False)
print("saved reports/lstm_results.csv")

parameters 484,993 | validation CCC 0.852 | TEST CCC 0.857 | Pearson 0.864 | MAE 0.054
saved reports/lstm_results.csv
